# EA-TTM — TTM Head Probing on Retail Demand

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 04 — Head-probed TTM (`HP-TTM`)  
**Scope:** Adapt only the Slim TTM Decoder and Forecasting Head to the M5 retail domain while keeping the pre-trained TTM backbone frozen.

The resulting model becomes the retail-adapted base forecaster used later by EA-TTM.

## Training contract

Let the pre-trained TTM be decomposed as:

$$
f_{\mathrm{TTM}}
=
f_{\mathrm{head}}
\circ
f_{\mathrm{decoder}}
\circ
f_{\mathrm{backbone}}.
$$

Head probing freezes the backbone:

$$
\nabla_{\theta_b}\mathcal{L}=0,
$$

while updating only decoder and forecasting-head parameters:

$$
\theta_d,\theta_h
\leftarrow
\operatorname*{arg\,min}_{\theta_d,\theta_h}
\mathcal{L}_{\mathrm{MSE}}.
$$

For a batch of 28-step retail targets:

$$
\mathcal{L}_{\mathrm{MSE}}
=
\frac{1}{B H}
\sum_{i=1}^{B}
\sum_{h=1}^{H}
\left(
y_{i,h}-\hat y_{i,h}
\right)^2,
\qquad
H=28.
$$

## Data protocol

The four pre-registered training origins are:

$$
\mathcal{O}_{train}
=
\{512,\ 960,\ 1409,\ 1857\}.
$$

For each store-item series, every origin contributes one 512-day context and one 28-day target. Therefore:

$$
30{,}490 \times 4
=
121{,}960
$$

training windows.

Validation remains fixed at:

$$
d_{1886}:d_{1913}.
$$

The locked final-test interval $d_{1914}:d_{1941}$ is not used in this notebook.

## Optimization strategy

To avoid arbitrarily guessing the learning rate, the notebook uses the IBM TSFM learning-rate finder over a bounded range:

$$
10^{-6}
\le
\eta
\le
10^{-2}.
$$

The selected learning rate is then used with AdamW and a maximum of 10 epochs. Validation MSE controls early stopping.

Training is checkpointed periodically so another Colab session can resume from the most recent saved state.

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

## Runtime and project preflight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")
MODEL_SNAPSHOT_DIR = PROJECT_ROOT / "models" / "pretrained_ttm"
HEAD_PROBED_MODEL_DIR = PROJECT_ROOT / "models" / "head_probed_ttm"

PROCESSED_M5_DIR = PROJECT_ROOT / "data" / "processed" / "m5"
METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results" / "tables"

RUN_DIR = PROJECT_ROOT / "runs" / "04_head_probe"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"

for directory in (
    HEAD_PROBED_MODEL_DIR,
    RESULTS_TABLE_DIR,
    RUN_DIR,
    CHECKPOINT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

required_artifacts = [
    MODEL_SNAPSHOT_DIR / "config.json",
    METADATA_DIR / "01_TTM_MODEL_CONTRACT.json",
    METADATA_DIR / "02_DATA_MANIFEST.json",
    METADATA_DIR / "02_SPLIT_MANIFEST.json",
    METADATA_DIR / "03_RUN_SUMMARY.json",
    PROCESSED_M5_DIR / "sales_train_evaluation.parquet",
]

missing_artifacts = [
    str(path)
    for path in required_artifacts
    if not path.exists()
]

assert not missing_artifacts, (
    "Required upstream artifacts are missing: "
    + "; ".join(missing_artifacts)
)

Mounted at /content/drive


In [3]:
TSFM_DISTRIBUTION_VERSION = "0.3.9"
TSFM_GIT_REPOSITORY = "https://github.com/ibm-granite/granite-tsfm.git"
TSFM_GIT_TAG = f"v{TSFM_DISTRIBUTION_VERSION}"
TSFM_SOURCE_LOCK = METADATA_DIR / "00_TSFM_SOURCE_LOCK.txt"

RUNTIME_VENDOR_ROOT = Path("/content/ea_ttm_vendor")
TSFM_SOURCE_ROOT = RUNTIME_VENDOR_ROOT / "granite-tsfm"
RUNTIME_VENDOR_ROOT.mkdir(parents=True, exist_ok=True)

def module_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

def run_command(command: list[str], cwd: Path | None = None) -> str:
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()

if not module_available("tsfm_public"):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            f"granite-tsfm=={TSFM_DISTRIBUTION_VERSION}",
        ],
        check=True,
    )
    importlib.invalidate_caches()

if not module_available("tsfm_public"):
    if TSFM_SOURCE_LOCK.exists():
        tsfm_source_commit = TSFM_SOURCE_LOCK.read_text(
            encoding="utf-8"
        ).strip()
    else:
        tag_query = subprocess.run(
            [
                "git",
                "ls-remote",
                "--tags",
                TSFM_GIT_REPOSITORY,
                f"refs/tags/{TSFM_GIT_TAG}",
            ],
            check=False,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if tag_query:
            tsfm_source_commit = tag_query.split()[0]
        else:
            tsfm_source_commit = run_command(
                [
                    "git",
                    "ls-remote",
                    TSFM_GIT_REPOSITORY,
                    "refs/heads/main",
                ]
            ).split()[0]

        TSFM_SOURCE_LOCK.write_text(
            tsfm_source_commit,
            encoding="utf-8",
        )

    if TSFM_SOURCE_ROOT.exists():
        subprocess.run(
            ["rm", "-rf", str(TSFM_SOURCE_ROOT)],
            check=True,
        )

    run_command(
        [
            "git",
            "clone",
            "--quiet",
            "--no-checkout",
            TSFM_GIT_REPOSITORY,
            str(TSFM_SOURCE_ROOT),
        ]
    )
    run_command(
        ["git", "checkout", "--quiet", tsfm_source_commit],
        cwd=TSFM_SOURCE_ROOT,
    )

    sys.path.insert(0, str(TSFM_SOURCE_ROOT))
    importlib.invalidate_caches()

assert module_available("tsfm_public")

from tsfm_public.models.tinytimemixer.modeling_tinytimemixer import (
    TinyTimeMixerForPrediction,
)
from tsfm_public.toolkit.lr_finder import optimal_lr_finder
from tsfm_public.toolkit.time_series_preprocessor import (
    DEFAULT_FREQUENCY_MAPPING,
)

## Reproducibility state

The project seed is fixed at:

$$
s=2026.
$$

Training order is deterministic by epoch:

$$
\pi_e
=
\operatorname{Permutation}
\left(
N;
s+e
\right).
$$

This lets a resumed run reconstruct the same sample ordering for any saved epoch.

In [4]:
BASE_SEED = 2026

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(True, warn_only=True)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

set_global_seed(BASE_SEED)

assert torch.cuda.is_available(), "CUDA GPU is required for head probing."
DEVICE = torch.device("cuda")

## Upstream contract verification

In [5]:
model_contract = json.loads(
    (METADATA_DIR / "01_TTM_MODEL_CONTRACT.json").read_text(
        encoding="utf-8"
    )
)

data_manifest = json.loads(
    (METADATA_DIR / "02_DATA_MANIFEST.json").read_text(
        encoding="utf-8"
    )
)

split_manifest = json.loads(
    (METADATA_DIR / "02_SPLIT_MANIFEST.json").read_text(
        encoding="utf-8"
    )
)

phase03_summary = json.loads(
    (METADATA_DIR / "03_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

CONTEXT_LENGTH = int(split_manifest["context_length"])
FORECAST_HORIZON = int(split_manifest["forecast_horizon"])

TRAIN_ORIGINS = [
    int(origin)
    for origin in split_manifest["train_origins"]
]

VALIDATION_ORIGIN = int(split_manifest["validation_origin"])
FREQUENCY = model_contract["requested_contract"]["frequency"]
FREQUENCY_TOKEN = int(
    DEFAULT_FREQUENCY_MAPPING[FREQUENCY]
)

DATASET_SHA256 = data_manifest["dataset_sha256"]
SPLIT_SHA256 = split_manifest["split_sha256"]

ZERO_SHOT_METRICS = phase03_summary["zero_shot_ttm_metrics"]

assert CONTEXT_LENGTH == 512
assert FORECAST_HORIZON == 28
assert TRAIN_ORIGINS == [512, 960, 1409, 1857]
assert VALIDATION_ORIGIN == 1885
assert split_manifest["locked_test_interval"] == ["d_1914", "d_1941"]

## Build deterministic retail training tensors

The notebook loads only the development period through `d_1913`. The final-test target is never materialized.

In [6]:
sales_path = PROCESSED_M5_DIR / "sales_train_evaluation.parquet"

identifier_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

development_columns = [
    f"d_{day}"
    for day in range(1, VALIDATION_ORIGIN + FORECAST_HORIZON + 1)
]

development_frame = pd.read_parquet(
    sales_path,
    columns=identifier_columns + development_columns,
)

series_count = int(development_frame.shape[0])

development_values = development_frame[
    development_columns
].to_numpy(
    dtype=np.float32,
    copy=True,
)

assert development_values.shape == (30_490, 1913)

training_contexts = np.empty(
    (
        series_count * len(TRAIN_ORIGINS),
        CONTEXT_LENGTH,
    ),
    dtype=np.float32,
)

training_targets = np.empty(
    (
        series_count * len(TRAIN_ORIGINS),
        FORECAST_HORIZON,
    ),
    dtype=np.float32,
)

training_origin_ids = np.empty(
    series_count * len(TRAIN_ORIGINS),
    dtype=np.int32,
)

for origin_index, origin in enumerate(TRAIN_ORIGINS):
    destination_start = origin_index * series_count
    destination_end = destination_start + series_count

    context_start = origin - CONTEXT_LENGTH
    context_end = origin

    target_start = origin
    target_end = origin + FORECAST_HORIZON

    training_contexts[
        destination_start:destination_end
    ] = development_values[
        :,
        context_start:context_end,
    ]

    training_targets[
        destination_start:destination_end
    ] = development_values[
        :,
        target_start:target_end,
    ]

    training_origin_ids[
        destination_start:destination_end
    ] = origin

validation_context = development_values[
    :,
    VALIDATION_ORIGIN - CONTEXT_LENGTH:VALIDATION_ORIGIN,
]

validation_target = development_values[
    :,
    VALIDATION_ORIGIN:VALIDATION_ORIGIN + FORECAST_HORIZON,
]

assert training_contexts.shape == (121_960, 512)
assert training_targets.shape == (121_960, 28)
assert validation_context.shape == (30_490, 512)
assert validation_target.shape == (30_490, 28)

print("Training contexts:", training_contexts.shape)
print("Training targets:", training_targets.shape)
print("Validation context:", validation_context.shape)
print("Validation target:", validation_target.shape)

Training contexts: (121960, 512)
Training targets: (121960, 28)
Validation context: (30490, 512)
Validation target: (30490, 28)


## Head-probing parameter contract

Only decoder and forecasting-head parameters are trainable:

$$
\theta_{\mathrm{train}}
=
\theta_{\mathrm{decoder}}
\cup
\theta_{\mathrm{head}}.
$$

The backbone is frozen:

$$
\theta_{\mathrm{backbone}}
\notin
\theta_{\mathrm{train}}.
$$

In [7]:
head_probe_model = TinyTimeMixerForPrediction.from_pretrained(
    MODEL_SNAPSHOT_DIR
).to(DEVICE)

for parameter in head_probe_model.parameters():
    parameter.requires_grad_(False)

for parameter in head_probe_model.decoder.parameters():
    parameter.requires_grad_(True)

for parameter in head_probe_model.head.parameters():
    parameter.requires_grad_(True)

total_parameter_count = sum(
    parameter.numel()
    for parameter in head_probe_model.parameters()
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in head_probe_model.parameters()
    if parameter.requires_grad
)

backbone_trainable_parameter_count = sum(
    parameter.numel()
    for parameter in head_probe_model.backbone.parameters()
    if parameter.requires_grad
)

decoder_trainable_parameter_count = sum(
    parameter.numel()
    for parameter in head_probe_model.decoder.parameters()
    if parameter.requires_grad
)

head_trainable_parameter_count = sum(
    parameter.numel()
    for parameter in head_probe_model.head.parameters()
    if parameter.requires_grad
)

assert backbone_trainable_parameter_count == 0
assert decoder_trainable_parameter_count > 0
assert head_trainable_parameter_count > 0
assert trainable_parameter_count == (
    decoder_trainable_parameter_count
    + head_trainable_parameter_count
)

print("Total parameters:", total_parameter_count)
print("Trainable parameters:", trainable_parameter_count)
print("Decoder trainable:", decoder_trainable_parameter_count)
print("Head trainable:", head_trainable_parameter_count)

Loading weights:   0%|          | 0/139 [00:00<?, ?it/s]

Total parameters: 799756
Trainable parameters: 246818
Decoder trainable: 191474
Head trainable: 55344


## Training dataset interface

Each sample contains only inputs accepted by the TTM forward method.

In [8]:
class HeadProbeDataset(Dataset):
    def __init__(
        self,
        contexts: np.ndarray,
        targets: np.ndarray,
        frequency_token: int,
    ) -> None:
        self.contexts = contexts
        self.targets = targets
        self.frequency_token = int(frequency_token)

    def __len__(self) -> int:
        return self.contexts.shape[0]

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {
            "past_values": torch.from_numpy(
                self.contexts[index]
            ).unsqueeze(-1),
            "future_values": torch.from_numpy(
                self.targets[index]
            ).unsqueeze(-1),
            "freq_token": torch.tensor(
                self.frequency_token,
                dtype=torch.long,
            ),
        }

training_dataset = HeadProbeDataset(
    training_contexts,
    training_targets,
    FREQUENCY_TOKEN,
)

## Training batch-size audit

The training batch size is selected from:

$$
\mathcal{B}_{train}
=
\{128,\ 256,\ 512\}.
$$

A candidate passes only if it completes a mixed-precision forward and backward pass with the backbone frozen.

In [9]:
TRAINING_BATCH_CANDIDATES = [128, 256, 512]

loss_function = torch.nn.MSELoss()

training_batch_audit_records: list[dict[str, Any]] = []
selected_training_batch_size = None

head_probe_model.train()

for candidate_batch_size in TRAINING_BATCH_CANDIDATES:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    batch_context = torch.from_numpy(
        training_contexts[:candidate_batch_size]
    ).unsqueeze(-1).to(DEVICE)

    batch_target = torch.from_numpy(
        training_targets[:candidate_batch_size]
    ).unsqueeze(-1).to(DEVICE)

    batch_frequency_token = torch.full(
        (candidate_batch_size,),
        FREQUENCY_TOKEN,
        dtype=torch.long,
        device=DEVICE,
    )

    try:
        head_probe_model.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            output = head_probe_model(
                past_values=batch_context,
                freq_token=batch_frequency_token,
                return_dict=True,
            )

            prediction = output.prediction_outputs
            loss = loss_function(
                prediction,
                batch_target,
            )

        loss.backward()
        torch.cuda.synchronize()

        peak_memory_gb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )

        training_batch_audit_records.append(
            {
                "batch_size": candidate_batch_size,
                "status": "passed",
                "peak_memory_gb": float(peak_memory_gb),
            }
        )

        selected_training_batch_size = candidate_batch_size

    except RuntimeError as error:
        if "out of memory" not in str(error).lower():
            raise

        training_batch_audit_records.append(
            {
                "batch_size": candidate_batch_size,
                "status": "cuda_oom",
                "peak_memory_gb": None,
            }
        )

        torch.cuda.empty_cache()
        break

    finally:
        head_probe_model.zero_grad(set_to_none=True)
        del batch_context, batch_target, batch_frequency_token

assert selected_training_batch_size is not None

training_batch_audit = pd.DataFrame(
    training_batch_audit_records
)

training_batch_audit

,batch_size,status,peak_memory_gb
0,128,passed,0.037304
1,256,passed,0.054913
2,512,passed,0.090740


## Learning-rate selection

The IBM TSFM learning-rate finder searches a bounded interval:

$$
10^{-6}
\rightarrow
10^{-2}
$$

over 100 updates and restores the original model weights afterward.

If an interrupted training checkpoint already exists, this search is skipped and the saved learning rate is reused.

In [10]:
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_training_state.pt"
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_validation_state.pt"

MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2
CHECKPOINT_INTERVAL_STEPS = 50

selected_learning_rate = None

if not LAST_CHECKPOINT_PATH.exists():
    lr_optimizer = torch.optim.AdamW(
        [
            parameter
            for parameter in head_probe_model.parameters()
            if parameter.requires_grad
        ],
        lr=1e-4,
    )

    selected_learning_rate, head_probe_model = optimal_lr_finder(
        model=head_probe_model,
        dset_train=training_dataset,
        optimizer=lr_optimizer,
        device="cuda",
        start_lr=1e-6,
        end_lr=1e-2,
        num_iter=100,
        batch_size=selected_training_batch_size,
        shuffle=True,
        step_mode="exp",
        suggestion="valley",
        plot_lr_finder=False,
        enable_prefix_tuning=True,
    )

    selected_learning_rate = float(selected_learning_rate)

    assert np.isfinite(selected_learning_rate)
    assert 1e-6 <= selected_learning_rate <= 1e-2

    del lr_optimizer

print("Selected learning rate:", selected_learning_rate)

Selected learning rate: 8.697490026177834e-05


## Checkpoint state

A recoverable training checkpoint stores:

- model state;
- optimizer state;
- AMP scaler state;
- epoch;
- next batch index within the epoch;
- global optimization step;
- best validation MSE;
- early-stopping counter;
- selected learning rate;
- project dataset/split fingerprints;
- Python, NumPy, CPU, and CUDA RNG states.

In [11]:
def capture_rng_state() -> dict[str, Any]:
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all(),
    }

def restore_rng_state(state: dict[str, Any]) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    torch.cuda.set_rng_state_all(state["torch_cuda"])

training_config = {
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "seed": BASE_SEED,
    "train_origins": TRAIN_ORIGINS,
    "validation_origin": VALIDATION_ORIGIN,
    "context_length": CONTEXT_LENGTH,
    "forecast_horizon": FORECAST_HORIZON,
    "frequency": FREQUENCY,
    "frequency_token": FREQUENCY_TOKEN,
    "training_batch_size": selected_training_batch_size,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "checkpoint_interval_steps": CHECKPOINT_INTERVAL_STEPS,
    "loss": "mse",
    "optimizer": "AdamW",
    "trainable_components": [
        "decoder",
        "head",
    ],
    "frozen_components": [
        "backbone",
    ],
}

training_config_sha256 = hashlib.sha256(
    json.dumps(
        training_config,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

## Validation function

Validation uses the same 28-day metric definitions as Phase 03 and applies no output clipping.

In [12]:
def forecasting_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    actual_64 = actual.astype(np.float64, copy=False)
    predicted_64 = predicted.astype(np.float64, copy=False)

    error = predicted_64 - actual_64
    absolute_error = np.abs(error)
    squared_error = np.square(error)

    mse = float(squared_error.mean())
    rmse = float(np.sqrt(mse))
    mae = float(absolute_error.mean())

    denominator = float(np.abs(actual_64).sum())
    assert denominator > 0.0

    wape = float(
        absolute_error.sum()
        / denominator
    )

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "wape": wape,
    }

def predict_validation(
    model: torch.nn.Module,
    batch_size: int,
) -> np.ndarray:
    model.eval()

    predictions = np.empty(
        (
            validation_context.shape[0],
            FORECAST_HORIZON,
        ),
        dtype=np.float32,
    )

    with torch.no_grad():
        for start in range(
            0,
            validation_context.shape[0],
            batch_size,
        ):
            end = min(
                start + batch_size,
                validation_context.shape[0],
            )

            batch_values = torch.from_numpy(
                validation_context[start:end]
            ).unsqueeze(-1).to(DEVICE)

            batch_token = torch.full(
                (end - start,),
                FREQUENCY_TOKEN,
                dtype=torch.long,
                device=DEVICE,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                output = model(
                    past_values=batch_values,
                    freq_token=batch_token,
                    return_dict=True,
                )

            predictions[start:end] = (
                output.prediction_outputs
                .squeeze(-1)
                .float()
                .cpu()
                .numpy()
            )

            del batch_values, batch_token, output

    return predictions

## Initialize or resume optimization

In [13]:
trainable_parameters = [
    parameter
    for parameter in head_probe_model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=selected_learning_rate if selected_learning_rate is not None else 1e-4,
)

scaler = torch.amp.GradScaler("cuda")

start_epoch = 0
start_batch_index = 0
global_step = 0
best_validation_mse = float("inf")
early_stopping_counter = 0

if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    assert checkpoint["training_config_sha256"] == training_config_sha256
    assert checkpoint["dataset_sha256"] == DATASET_SHA256
    assert checkpoint["split_sha256"] == SPLIT_SHA256

    selected_learning_rate = float(
        checkpoint["selected_learning_rate"]
    )

    for param_group in optimizer.param_groups:
        param_group["lr"] = selected_learning_rate

    head_probe_model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )
    scaler.load_state_dict(
        checkpoint["scaler_state_dict"]
    )

    start_epoch = int(checkpoint["epoch"])
    start_batch_index = int(
        checkpoint["next_batch_index"]
    )
    global_step = int(checkpoint["global_step"])
    best_validation_mse = float(
        checkpoint["best_validation_mse"]
    )
    early_stopping_counter = int(
        checkpoint["early_stopping_counter"]
    )

    restore_rng_state(
        checkpoint["rng_state"]
    )

print("Training starts at epoch:", start_epoch)
print("Training starts at batch:", start_batch_index)
print("Global step:", global_step)
print("Learning rate:", selected_learning_rate)

Training starts at epoch: 0
Training starts at batch: 0
Global step: 0
Learning rate: 8.697490026177834e-05


## Head-probing training loop

The backbone remains in evaluation mode during optimization. Only decoder and head parameters receive gradients.

In [14]:
training_history_records: list[dict[str, Any]] = []

training_start_time = time.perf_counter()
training_completed = False

num_training_samples = training_contexts.shape[0]
num_batches_per_epoch = math.ceil(
    num_training_samples
    / selected_training_batch_size
)

def save_training_checkpoint(
    *,
    path: Path,
    epoch: int,
    next_batch_index: int,
    best_mse: float,
    patience_counter: int,
) -> None:
    checkpoint_payload = {
        "model_state_dict": head_probe_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "epoch": int(epoch),
        "next_batch_index": int(next_batch_index),
        "global_step": int(global_step),
        "best_validation_mse": float(best_mse),
        "early_stopping_counter": int(patience_counter),
        "selected_learning_rate": float(selected_learning_rate),
        "training_config_sha256": training_config_sha256,
        "dataset_sha256": DATASET_SHA256,
        "split_sha256": SPLIT_SHA256,
        "rng_state": capture_rng_state(),
    }

    temporary_path = path.with_suffix(".tmp")
    torch.save(
        checkpoint_payload,
        temporary_path,
    )
    temporary_path.replace(path)

for epoch in range(start_epoch, MAX_EPOCHS):
    head_probe_model.train()
    head_probe_model.backbone.eval()

    epoch_rng = np.random.default_rng(
        BASE_SEED + epoch
    )

    epoch_permutation = epoch_rng.permutation(
        num_training_samples
    )

    epoch_loss_sum = 0.0
    epoch_observation_count = 0

    first_batch = (
        start_batch_index
        if epoch == start_epoch
        else 0
    )

    for batch_index in range(
        first_batch,
        num_batches_per_epoch,
    ):
        batch_start = (
            batch_index
            * selected_training_batch_size
        )
        batch_end = min(
            batch_start
            + selected_training_batch_size,
            num_training_samples,
        )

        sample_indices = epoch_permutation[
            batch_start:batch_end
        ]

        batch_context = torch.from_numpy(
            training_contexts[sample_indices]
        ).unsqueeze(-1).to(DEVICE)

        batch_target = torch.from_numpy(
            training_targets[sample_indices]
        ).unsqueeze(-1).to(DEVICE)

        batch_token = torch.full(
            (batch_end - batch_start,),
            FREQUENCY_TOKEN,
            dtype=torch.long,
            device=DEVICE,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            output = head_probe_model(
                past_values=batch_context,
                freq_token=batch_token,
                return_dict=True,
            )

            prediction = output.prediction_outputs

            loss = loss_function(
                prediction,
                batch_target,
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        current_batch_size = (
            batch_end - batch_start
        )

        epoch_loss_sum += (
            float(loss.detach().item())
            * current_batch_size
        )
        epoch_observation_count += current_batch_size

        global_step += 1

        if (
            global_step
            % CHECKPOINT_INTERVAL_STEPS
            == 0
        ):
            save_training_checkpoint(
                path=LAST_CHECKPOINT_PATH,
                epoch=epoch,
                next_batch_index=batch_index + 1,
                best_mse=best_validation_mse,
                patience_counter=early_stopping_counter,
            )

        del (
            batch_context,
            batch_target,
            batch_token,
            output,
            prediction,
            loss,
        )

    mean_training_loss = (
        epoch_loss_sum
        / epoch_observation_count
    )

    validation_prediction = predict_validation(
        head_probe_model,
        selected_training_batch_size,
    )

    validation_metrics = forecasting_metrics(
        validation_target,
        validation_prediction,
    )

    training_history_records.append(
        {
            "epoch": epoch + 1,
            "train_mse": mean_training_loss,
            "validation_mse": validation_metrics["mse"],
            "validation_rmse": validation_metrics["rmse"],
            "validation_mae": validation_metrics["mae"],
            "validation_wape": validation_metrics["wape"],
            "global_step": global_step,
        }
    )

    improved = (
        validation_metrics["mse"]
        < best_validation_mse
    )

    if improved:
        best_validation_mse = float(
            validation_metrics["mse"]
        )
        early_stopping_counter = 0

        save_training_checkpoint(
            path=BEST_CHECKPOINT_PATH,
            epoch=epoch + 1,
            next_batch_index=0,
            best_mse=best_validation_mse,
            patience_counter=early_stopping_counter,
        )
    else:
        early_stopping_counter += 1

    save_training_checkpoint(
        path=LAST_CHECKPOINT_PATH,
        epoch=epoch + 1,
        next_batch_index=0,
        best_mse=best_validation_mse,
        patience_counter=early_stopping_counter,
    )

    start_batch_index = 0

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train MSE={mean_training_loss:.6f} | "
        f"val MSE={validation_metrics['mse']:.6f} | "
        f"val WAPE={validation_metrics['wape']:.6f}"
    )

    if (
        early_stopping_counter
        >= EARLY_STOPPING_PATIENCE
    ):
        break

training_completed = True
training_runtime_seconds = (
    time.perf_counter()
    - training_start_time
)

assert BEST_CHECKPOINT_PATH.exists()

Epoch 01 | train MSE=5.727579 | val MSE=4.441949 | val WAPE=0.721398
Epoch 02 | train MSE=5.602208 | val MSE=4.422208 | val WAPE=0.723701
Epoch 03 | train MSE=5.501520 | val MSE=4.430881 | val WAPE=0.723540
Epoch 04 | train MSE=5.425839 | val MSE=4.410101 | val WAPE=0.719494
Epoch 05 | train MSE=5.233175 | val MSE=4.404917 | val WAPE=0.722935
Epoch 06 | train MSE=5.156290 | val MSE=4.381235 | val WAPE=0.726578
Epoch 07 | train MSE=5.156806 | val MSE=4.406647 | val WAPE=0.721625
Epoch 08 | train MSE=5.088933 | val MSE=4.406917 | val WAPE=0.725934


## Restore the best validation checkpoint and freeze HP-TTM

In [15]:
best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

head_probe_model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

for parameter in head_probe_model.parameters():
    parameter.requires_grad_(False)

head_probe_model.eval()

assert sum(
    parameter.numel()
    for parameter in head_probe_model.parameters()
    if parameter.requires_grad
) == 0

best_validation_prediction = predict_validation(
    head_probe_model,
    selected_training_batch_size,
)

best_head_probe_metrics = forecasting_metrics(
    validation_target,
    best_validation_prediction,
)

head_probe_model.save_pretrained(
    HEAD_PROBED_MODEL_DIR
)

assert (
    HEAD_PROBED_MODEL_DIR
    / "config.json"
).exists()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Benchmark comparison

The comparison isolates the effect of target-domain head probing:

- Zero-shot TTM: no retail-domain training.
- HP-TTM: same pre-trained backbone, frozen; decoder and head adapted to retail.

In [16]:
def percent_change(
    baseline: float,
    candidate: float,
) -> float:
    return float(
        100.0
        * (baseline - candidate)
        / baseline
    )

comparison = {
    "mse_improvement_percent": percent_change(
        ZERO_SHOT_METRICS["mse"],
        best_head_probe_metrics["mse"],
    ),
    "rmse_improvement_percent": percent_change(
        ZERO_SHOT_METRICS["rmse"],
        best_head_probe_metrics["rmse"],
    ),
    "mae_improvement_percent": percent_change(
        ZERO_SHOT_METRICS["mae"],
        best_head_probe_metrics["mae"],
    ),
    "wape_improvement_percent": percent_change(
        ZERO_SHOT_METRICS["wape"],
        best_head_probe_metrics["wape"],
    ),
}

comparison_table = pd.DataFrame(
    [
        {
            "model": "Zero-shot TTM",
            **ZERO_SHOT_METRICS,
        },
        {
            "model": "Head-probed TTM",
            **best_head_probe_metrics,
        },
    ]
)

comparison_table

,model,mse,rmse,mae,wape
0,Zero-shot TTM,4.557264,2.134775,1.021532,0.736806
1,Head-probed TTM,4.381236,2.093140,1.007352,0.726578


## Persist Phase 04 artifacts

In [17]:
TRAINING_HISTORY_PATH = (
    RESULTS_TABLE_DIR
    / "04_HEAD_PROBE_TRAINING_HISTORY.csv"
)
COMPARISON_PATH = (
    RESULTS_TABLE_DIR
    / "04_HEAD_PROBE_VALIDATION_COMPARISON.csv"
)
BATCH_AUDIT_PATH = (
    RESULTS_TABLE_DIR
    / "04_TRAINING_BATCH_AUDIT.csv"
)
VALIDATION_PREDICTIONS_PATH = (
    RUN_DIR
    / "04_HEAD_PROBE_VALIDATION_PREDICTIONS.parquet"
)
RUN_SUMMARY_PATH = (
    METADATA_DIR
    / "04_RUN_SUMMARY.json"
)

pd.DataFrame(
    training_history_records
).to_csv(
    TRAINING_HISTORY_PATH,
    index=False,
)

comparison_table.to_csv(
    COMPARISON_PATH,
    index=False,
)

training_batch_audit.to_csv(
    BATCH_AUDIT_PATH,
    index=False,
)

prediction_frame = development_frame[
    identifier_columns
].copy()

for horizon_index in range(
    FORECAST_HORIZON
):
    horizon_number = horizon_index + 1

    prediction_frame[
        f"actual_h{horizon_number:02d}"
    ] = validation_target[:, horizon_index]

    prediction_frame[
        f"head_probe_h{horizon_number:02d}"
    ] = best_validation_prediction[
        :,
        horizon_index,
    ]

prediction_frame.to_parquet(
    VALIDATION_PREDICTIONS_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

best_epoch = int(
    best_checkpoint["epoch"]
)

run_summary = {
    "status": "PHASE_04_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "training_config_sha256": training_config_sha256,
    "train_origins": TRAIN_ORIGINS,
    "validation_target": [
        "d_1886",
        "d_1913",
    ],
    "selected_training_batch_size": int(
        selected_training_batch_size
    ),
    "selected_learning_rate": float(
        selected_learning_rate
    ),
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "best_epoch": best_epoch,
    "global_step": int(global_step),
    "training_runtime_seconds": float(
        training_runtime_seconds
    ),
    "total_parameters": int(
        total_parameter_count
    ),
    "trainable_parameters": int(
        trainable_parameter_count
    ),
    "backbone_trainable_parameters": int(
        backbone_trainable_parameter_count
    ),
    "decoder_trainable_parameters": int(
        decoder_trainable_parameter_count
    ),
    "head_trainable_parameters": int(
        head_trainable_parameter_count
    ),
    "zero_shot_metrics": ZERO_SHOT_METRICS,
    "head_probed_metrics": best_head_probe_metrics,
    "head_probe_vs_zero_shot": comparison,
}

RUN_SUMMARY_PATH.write_text(
    json.dumps(run_summary, indent=2),
    encoding="utf-8",
)

1390

In [18]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_lines = [
    "# EA-TTM Project Status",
    "",
    "## Current stage",
    "Phase 04 — Head-probed TTM completed and frozen.",
    "",
    "## HP-TTM training contract",
    f"- Training origins: {TRAIN_ORIGINS}",
    f"- Batch size: {selected_training_batch_size}",
    f"- Learning rate: {selected_learning_rate}",
    f"- Best epoch: {best_epoch}",
    f"- Trainable parameters: {trainable_parameter_count:,}",
    "- Frozen component during training: backbone",
    "- Trainable components during training: decoder + head",
    "- Complete HP-TTM frozen after model selection",
    "",
    "## Validation comparison",
    f"- Zero-shot MSE: {ZERO_SHOT_METRICS['mse']:.6f}",
    f"- HP-TTM MSE: {best_head_probe_metrics['mse']:.6f}",
    f"- Zero-shot WAPE: {ZERO_SHOT_METRICS['wape']:.6f}",
    f"- HP-TTM WAPE: {best_head_probe_metrics['wape']:.6f}",
    "",
    "## Next phase",
    "Audit the Slim Decoder feature extraction from the frozen HP-TTM and define",
    "the explicit decoder-to-adapter cross-feed contract for EA-TTM.",
]

status_path.write_text(
    "\n".join(status_lines),
    encoding="utf-8",
)

695

## Phase validation

Phase 04 passes only if:

$$
\begin{aligned}
&\text{backbone trainable parameters}=0
\\
&\land\ \text{decoder trainable parameters}>0
\\
&\land\ \text{head trainable parameters}>0
\\
&\land\ \text{best validation checkpoint exists}
\\
&\land\ \text{HP-TTM snapshot exists}
\\
&\land\ \text{the complete selected HP-TTM is frozen}.
\end{aligned}
$$

In [19]:
phase_validation = {
    "backbone_frozen_during_training": (
        backbone_trainable_parameter_count == 0
    ),
    "decoder_was_trainable": (
        decoder_trainable_parameter_count > 0
    ),
    "head_was_trainable": (
        head_trainable_parameter_count > 0
    ),
    "best_checkpoint_exists": (
        BEST_CHECKPOINT_PATH.exists()
    ),
    "head_probed_snapshot_exists": (
        HEAD_PROBED_MODEL_DIR
        / "config.json"
    ).exists(),
    "selected_model_fully_frozen": (
        sum(
            parameter.numel()
            for parameter in head_probe_model.parameters()
            if parameter.requires_grad
        )
        == 0
    ),
    "run_summary_exists": RUN_SUMMARY_PATH.exists(),
}

assert all(phase_validation.values()), (
    f"Phase 04 validation failed: {phase_validation}"
)

phase_validation

{'backbone_frozen_during_training': True,
 'decoder_was_trainable': True,
 'head_was_trainable': True,
 'best_checkpoint_exists': True,
 'head_probed_snapshot_exists': True,
 'selected_model_fully_frozen': True,
 'run_summary_exists': True}

## Return block

The final block is the **only successful-run output that needs to be returned to ChatGPT**.

In [20]:
return_block = {
    "status": "PHASE_04_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "training_config_sha256": training_config_sha256,
    "train_origins": TRAIN_ORIGINS,
    "validation_target": [
        "d_1886",
        "d_1913",
    ],
    "selected_training_batch_size": int(
        selected_training_batch_size
    ),
    "selected_learning_rate": float(
        selected_learning_rate
    ),
    "best_epoch": best_epoch,
    "global_step": int(global_step),
    "training_runtime_seconds": float(
        training_runtime_seconds
    ),
    "parameter_counts": {
        "total": int(total_parameter_count),
        "trainable_during_head_probe": int(
            trainable_parameter_count
        ),
        "backbone_trainable": int(
            backbone_trainable_parameter_count
        ),
        "decoder_trainable": int(
            decoder_trainable_parameter_count
        ),
        "head_trainable": int(
            head_trainable_parameter_count
        ),
    },
    "zero_shot_metrics": ZERO_SHOT_METRICS,
    "head_probed_metrics": best_head_probe_metrics,
    "head_probe_vs_zero_shot": comparison,
    "selected_head_probed_model_frozen": True,
    "artifacts": {
        "head_probed_model": str(
            HEAD_PROBED_MODEL_DIR
        ),
        "last_checkpoint": str(
            LAST_CHECKPOINT_PATH
        ),
        "best_checkpoint": str(
            BEST_CHECKPOINT_PATH
        ),
        "training_history": str(
            TRAINING_HISTORY_PATH
        ),
        "validation_comparison": str(
            COMPARISON_PATH
        ),
        "training_batch_audit": str(
            BATCH_AUDIT_PATH
        ),
        "validation_predictions": str(
            VALIDATION_PREDICTIONS_PATH
        ),
        "run_summary": str(
            RUN_SUMMARY_PATH
        ),
        "project_status": str(
            status_path
        ),
    },
}

print("PHASE 04 RETURN BLOCK")
print(json.dumps(return_block, indent=2))

PHASE 04 RETURN BLOCK
{
  "status": "PHASE_04_COMPLETE",
  "dataset_sha256": "83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59",
  "split_sha256": "5ff2e30f43a832bb9e513153bb15daa4e1e67c871b43586f2f2f11ef51396447",
  "training_config_sha256": "8f511e17136b0eff12c826033d4877ac73ad770d8e491d36affd6f1da3ebc885",
  "train_origins": [
    512,
    960,
    1409,
    1857
  ],
  "validation_target": [
    "d_1886",
    "d_1913"
  ],
  "selected_training_batch_size": 512,
  "selected_learning_rate": 8.697490026177834e-05,
  "best_epoch": 6,
  "global_step": 1912,
  "training_runtime_seconds": 55.51601339599995,
  "parameter_counts": {
    "total": 799756,
    "trainable_during_head_probe": 246818,
    "backbone_trainable": 0,
    "decoder_trainable": 191474,
    "head_trainable": 55344
  },
  "zero_shot_metrics": {
    "mse": 4.557263813520326,
    "rmse": 2.1347748859119373,
    "mae": 1.0215323422208904,
    "wape": 0.7368058755052851
  },
  "head_probed_metrics": {
    "mse